# 04 — PPO Alignment

Fine-tunes the SFT model with Proximal Policy Optimization (PPO), using the
reward model from `03_reward_model.ipynb` as the scoring signal.
This is the core RLHF loop: generate bios, score them, update the policy.

Requires: SFT checkpoint (`results/sft_checkpoint`) and reward model checkpoint
(`results/reward_checkpoint`) from previous notebooks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/aipi590-challenge-2/blob/main/notebooks/04_ppo.ipynb)

In [ ]:
# Setup — install dependencies, clone repo, configure environment
!pip install -q trl peft transformers datasets accelerate bitsandbytes

import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import sys
sys.path.insert(0, "/content/aipi590-challenge-2")

from src.colab_utils import prepare_notebook, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")
print(f"Repo root: {repo_root}")

# Verify required checkpoints
sft_checkpoint = paths["results"] / "sft_checkpoint"
rm_checkpoint  = paths["results"] / "reward_checkpoint"

for ckpt, name in [(sft_checkpoint, "SFT"), (rm_checkpoint, "Reward")]:
    if not ckpt.exists():
        print(f"WARNING: {name} checkpoint not found at {ckpt}")
    else:
        print(f"{name} checkpoint found: {ckpt}")

In [ ]:
# Load SFT policy model and reward model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Policy model (SFT checkpoint wrapped with value head for PPO)
policy_base = str(sft_checkpoint) if sft_checkpoint.exists() else "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(policy_base)
tokenizer.pad_token = tokenizer.eos_token

policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(policy_base)
policy_model = policy_model.to(device)

# Reward model (frozen — used only for scoring)
rm_base = str(rm_checkpoint) if rm_checkpoint.exists() else "distilgpt2"
reward_model = AutoModelForSequenceClassification.from_pretrained(rm_base, num_labels=1)
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model = reward_model.to(device).eval()
for param in reward_model.parameters():
    param.requires_grad = False

print("Models loaded.")

In [ ]:
# PPO training loop
import torch
import json
from pathlib import Path

PROMPTS = [
    "Software engineer, 28, NYC, climbs on weekends, makes sourdough",
    "Nurse, 26, Chicago, reads sci-fi, runs half marathons",
    "High school history teacher, 31, Austin, plays guitar, homebrews beer",
    "UX designer, 27, SF, surfs, obsessed with typography",
    "Grad student in ecology, 25, Seattle, birdwatcher, plays chess",
    "Pastry chef, 29, New Orleans, jazz fan, does pottery",
    "Aerospace engineer, 30, LA, amateur astronomer, plays volleyball",
    "Documentary filmmaker, 27, Portland, vegan, forages for mushrooms",
    "Financial analyst, 32, Boston, into CrossFit, reads philosophy",
    "Veterinarian, 28, Denver, hikes fourteeners, plays piano",
    "Product manager, 34, NYC, salsa dancer, speaks four languages",
    "Marine biologist, 26, Miami, spearfishes, plays drums",
    "Civil engineer, 33, Phoenix, restores motorcycles, loves spicy food",
    "Librarian, 29, Minneapolis, writes poetry, into vintage fashion",
    "Data scientist, 27, SF, rock climbs, brews kombucha",
    "Architect, 35, Chicago, sketches cities, sails on Lake Michigan",
]

from src.dataset import format_sft_prompt

def score_bio(prompt_text: str, bio_text: str) -> float:
    """Score a (prompt, bio) pair with the frozen reward model."""
    full_text = prompt_text + bio_text
    enc = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(device)
    with torch.no_grad():
        out = reward_model(**enc)
    return out.logits.squeeze().item()


ppo_config = PPOConfig(
    batch_size=16,
    learning_rate=1e-5,
    kl_penalty="kl",
    init_kl_coef=0.2,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=None,
    tokenizer=tokenizer,
)

NUM_STEPS = 20
training_log = []

print(f"Starting PPO training for {NUM_STEPS} steps...")

for step in range(NUM_STEPS):
    # Build a batch of prompts (cycle through PROMPTS)
    batch_prompts = [PROMPTS[i % len(PROMPTS)] for i in range(step * ppo_config.batch_size, (step + 1) * ppo_config.batch_size)]
    formatted = [format_sft_prompt(p) for p in batch_prompts]

    # Tokenize queries
    query_tensors = [
        tokenizer(p, return_tensors="pt", truncation=True, max_length=128)["input_ids"].squeeze(0).to(device)
        for p in formatted
    ]

    # Generate responses
    response_tensors = ppo_trainer.generate(
        query_tensors,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode and score
    rewards = []
    for q_ids, r_ids, prompt_text in zip(query_tensors, response_tensors, formatted):
        response_text = tokenizer.decode(r_ids[len(q_ids):], skip_special_tokens=True)
        score = score_bio(prompt_text, response_text)
        rewards.append(torch.tensor(score, dtype=torch.float32))

    # PPO step
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    mean_reward = float(torch.stack(rewards).mean())
    kl = stats.get("objective/kl", float("nan"))
    training_log.append({"step": step, "mean_reward": mean_reward, "kl": kl})

    if step % 5 == 0 or step == NUM_STEPS - 1:
        print(f"Step {step:3d} | mean_reward={mean_reward:.4f} | kl={kl:.4f}")

print("PPO training complete.")

In [ ]:
# Plot training curves (KL divergence and mean reward per step)
import matplotlib.pyplot as plt
import numpy as np

steps        = [entry["step"] for entry in training_log]
mean_rewards = [entry["mean_reward"] for entry in training_log]
kl_values    = [entry["kl"] for entry in training_log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps, mean_rewards, color="#4A90D9", linewidth=2, marker="o", markersize=4)
ax1.set_title("Mean Reward per Step", fontsize=13, fontweight="bold")
ax1.set_xlabel("PPO Step", fontsize=11)
ax1.set_ylabel("Mean Reward", fontsize=11)
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(alpha=0.3, linestyle="--")

ax2.plot(steps, kl_values, color="#E07B54", linewidth=2, marker="o", markersize=4)
ax2.set_title("KL Divergence per Step", fontsize=13, fontweight="bold")
ax2.set_xlabel("PPO Step", fontsize=11)
ax2.set_ylabel("KL Divergence", fontsize=11)
ax2.spines[["top", "right"]].set_visible(False)
ax2.grid(alpha=0.3, linestyle="--")

plt.tight_layout()

charts_dir = paths["results"] / "metrics"
charts_dir.mkdir(parents=True, exist_ok=True)

chart_path = charts_dir / "ppo_training_curves.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {chart_path}")

In [ ]:
# Save PPO checkpoint, training log, and publish
import json

# Save policy model
ppo_checkpoint = paths["results"] / "ppo_checkpoint"
ppo_trainer.model.save_pretrained(str(ppo_checkpoint))
tokenizer.save_pretrained(str(ppo_checkpoint))
print(f"PPO model saved to {ppo_checkpoint}")

# Save training log
log_path = paths["results"] / "metrics" / "ppo_training_log.json"
log_path.write_text(json.dumps({
    "config": {
        "batch_size": ppo_config.batch_size,
        "learning_rate": ppo_config.learning_rate,
        "kl_penalty": ppo_config.kl_penalty,
        "init_kl_coef": ppo_config.init_kl_coef,
        "num_steps": NUM_STEPS,
    },
    "log": training_log,
}, indent=2))
print(f"Training log saved to {log_path}")

publish_artifacts(
    [log_path, paths["results"] / "metrics" / "ppo_training_curves.png"],
    "add PPO training log and curves",
    repo_root,
)
print("Done.")